In [ ]:
!pip install contractions spacy nltk pandas
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 96.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import re
import pandas as pd
import nltk
import spacy
import contractions
from google.colab import files

nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
try:
	df = pd.read_csv('my_data (1).csv')
except FileNotFoundError:
	uploaded = files.upload()
	df = pd.read_csv('my_data (1).csv')

print("Dataset Columns:", df.columns.tolist())

# Select the first object/string column as target, or explicitly name it
text_cols = df.select_dtypes(include=['object']).columns
if len(text_cols) > 0:
	target_col = text_cols[0]
	print(f"Automatically selected target column: '{target_col}'")
else:
	target_col = df.columns[0]
	print(f"Fallback selected target column: '{target_col}'")

Dataset Columns: ['name', 'city', 'gender', 'profession', 'age', 'cgpa', 'placed']
Automatically selected target column: 'name'


In [ ]:
def expand_text_contractions(text):
	return contractions.fix(text)

def clean_noise(text):
	text = text.lower()
	text = re.sub(r'https?://\S+|www\.\S+', '', text)
	text = re.sub(r'<.*?>', '', text)
	text = re.sub(r'[^a-zA-Z\s]', '', text)
	text = re.sub(r'\s+', ' ', text).strip()
	return text

def preprocess_text(text):
	if not isinstance(text, str):
		return []

	expanded = expand_text_contractions(text)
	cleaned = clean_noise(expanded)

	doc = nlp(cleaned)
	lemmatized_tokens = [
		token.lemma_ for token in doc
		if token.text not in stop_words and not token.is_space
	]

	return lemmatized_tokens

In [ ]:
df['cleaned_tokens'] = df[target_col].apply(preprocess_text)
df['processed_text'] = df['cleaned_tokens'].apply(lambda tokens: ' '.join(tokens))

In [ ]:
df[[target_col, 'processed_text', 'cleaned_tokens']]

,name,processed_text,cleaned_tokens
0,moriarity,moriarity,[moriarity]
1,moriarity,moriarity,[moriarity]
2,holmes,holme,[holme]
3,sam,sam,[sam]
4,sam,sam,[sam]
...,...,...,...
1095,holmes,holme,[holme]
1096,moriarity,moriarity,[moriarity]
1097,sam,sam,[sam]
1098,dean,dean,[dean]
